In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/fraud detection project/"
DATA = BASE          # CSVs are directly in this folder
DEV_MODE = True
DEV_ROWS = 250_000

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def reduce_mem(df):
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="float")
        elif pd.api.types.is_integer_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

trans = reduce_mem(pd.read_csv(DATA + "train_transaction.csv"))
ident = reduce_mem(pd.read_csv(DATA + "train_identity.csv"))

df = trans.merge(ident, on="TransactionID", how="left")   # left join
del trans, ident

df = df.sort_values("TransactionDT").reset_index(drop=True)  # time order
if DEV_MODE:
    df = df.head(DEV_ROWS).copy()   # first N rows by time, not random

print(df.shape)

(250000, 434)


In [ ]:
print(f"Fraud rate: {df['isFraud'].mean():.2%}")

missing = (df.isna().mean() * 100).sort_values(ascending=False)
print("\nTop missing (%):\n", missing.head(10).round(1))

print("\nUnique values in graph candidates:")
print(df[["card1","card2","addr1","P_emaildomain","DeviceInfo"]].nunique())

print("\nFraud rate by ProductCD:")
print(df.groupby("ProductCD")["isFraud"].mean().round(4))

Fraud rate: 3.25%

Top missing (%):
 id_24    98.9
id_25    98.9
id_08    98.9
id_07    98.9
id_21    98.9
id_26    98.8
id_27    98.8
id_22    98.8
id_23    98.8
D7       93.6
dtype: float64

Unique values in graph candidates:
card1            10547
card2              499
addr1              288
P_emaildomain       59
DeviceInfo        1274
dtype: int64

Fraud rate by ProductCD:
ProductCD
C    0.1061
H    0.0354
R    0.0255
S    0.0449
W    0.0199
Name: isFraud, dtype: float64


In [ ]:
df["day"] = df["TransactionDT"] // 86400
df["hour"] = (df["TransactionDT"] // 3600) % 24
df.to_parquet(BASE + "merged.parquet")   # saved to Drive, survives disconnects

In [ ]:
df = pd.read_parquet(BASE + "merged.parquet")

# Encode text columns as integers (no target used, so no leakage)
cat_cols = df.select_dtypes(include="object").columns
for c in cat_cols:
    df[c] = df[c].astype("category").cat.codes.replace(-1, np.nan)

# Never feed these to the model
drop_cols = ["isFraud", "TransactionID", "TransactionDT"]
features = [c for c in df.columns if c not in drop_cols]

# Drop columns that are more than 90% missing (little signal, big cost)
features = [c for c in features if df[c].isna().mean() < 0.90]

# Time-based split: train on the past, test on the future
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
X_tr, y_tr = train[features], train["isFraud"]
X_te, y_te = test[features], test["isFraud"]

print("features:", len(features))
print("train:", X_tr.shape, f"fraud {y_tr.mean():.2%}")
print("test: ", X_te.shape, f"fraud {y_te.mean():.2%}")

features: 421
train: (200000, 421) fraud 3.01%
test:  (50000, 421) fraud 4.22%


In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score)

results = {}
def evaluate(name, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    results[name] = {
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall":    recall_score(y_te, pred),
        "F1":        f1_score(y_te, pred),
        "PR-AUC":    average_precision_score(y_te, proba),
        "ROC-AUC":   roc_auc_score(y_te, proba),
    }
    return pd.DataFrame(results).T.round(4)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

# 1. Logistic Regression (needs imputing and scaling)
lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   LogisticRegression(class_weight="balanced", max_iter=300))
lr.fit(X_tr, y_tr)
evaluate("LogReg", lr.predict_proba(X_te)[:, 1])

# 2. Random Forest (needs no NaNs)
rf = RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1,
                            class_weight="balanced_subsample", random_state=42)
rf.fit(X_tr.fillna(-999), y_tr)
evaluate("RandomForest", rf.predict_proba(X_te.fillna(-999))[:, 1])

# 3. LightGBM (handles NaN natively)
spw = (y_tr == 0).sum() / (y_tr == 1).sum()
gbm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                         subsample=0.8, colsample_bytree=0.6,
                         scale_pos_weight=spw, random_state=42, verbose=-1)
gbm.fit(X_tr, y_tr)
evaluate("LightGBM", gbm.predict_proba(X_te)[:, 1])

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Precision,Recall,F1,PR-AUC,ROC-AUC
LogReg,0.1243,0.7899,0.2148,0.4373,0.8567
RandomForest,0.2916,0.6311,0.3989,0.5178,0.8786
LightGBM,0.4301,0.6524,0.5185,0.6263,0.9235


In [ ]:
print(pd.DataFrame(results).T.round(4))

import joblib
joblib.dump(features, BASE + "features_baseline.pkl")

              Precision  Recall      F1  PR-AUC  ROC-AUC
LogReg           0.1243  0.7899  0.2148  0.4373   0.8567
RandomForest     0.2916  0.6311  0.3989  0.5178   0.8786
LightGBM         0.4301  0.6524  0.5185  0.6263   0.9235


['/content/drive/MyDrive/fraud detection project/features_baseline.pkl']

In [ ]:
import networkx as nx

raw = pd.read_parquet(BASE + "merged.parquet")   # raw strings, same row order
n = len(raw); split = int(n * 0.8)
raw["card_node"] = "c_" + raw["card1"].astype(str)

# ---- 1. Build the graph ----
G = nx.Graph()
hub_limit = 0.01 * n
for col in ["addr1", "P_emaildomain", "DeviceInfo"]:
    cnt = raw[col].value_counts()
    e = raw[["card_node", col]].dropna().drop_duplicates()
    e = e[e[col].map(cnt) < hub_limit]
    G.add_edges_from(zip(e["card_node"], col + "_" + e[col].astype(str)))
print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

# ---- 2. Structural graph features ----
deg = dict(G.degree())
pr = nx.pagerank(G)
comms = nx.community.louvain_communities(G, seed=42)
comm_of = {node: i for i, c in enumerate(comms) for node in c}
comm_size = {node: len(comms[i]) for node, i in comm_of.items()}

gf = pd.DataFrame(index=raw.index)
gf["g_degree"] = raw["card_node"].map(deg)
gf["g_pagerank"] = raw["card_node"].map(pr)
gf["g_comm_size"] = raw["card_node"].map(comm_size)
raw["comm_id"] = raw["card_node"].map(comm_of)

# ---- 3. Velocity (uses only the past) ----
g = raw.groupby("card1")
gf["v_prior_tx"] = g.cumcount()
gf["v_secs_since_last"] = g["TransactionDT"].diff()

# ---- 4. Suspicious neighbors (label-safe) ----
# labels visible only for the train period
raw["_yk"] = raw["isFraud"].where(np.arange(n) < split, 0)
for col in ["card1", "addr1", "P_emaildomain", "DeviceInfo", "comm_id"]:
    gg = raw.groupby(col)
    gf[f"prior_fraud_{col}"] = gg["_yk"].cumsum() - raw["_yk"]   # earlier labels only
    gf[f"prior_tx_{col}"] = gg.cumcount()
    gf[f"prior_rate_{col}"] = gf[f"prior_fraud_{col}"] / gf[f"prior_tx_{col}"].replace(0, np.nan)

print(gf.shape)
gf.describe().T[["mean", "max"]].head(8)

nodes: 7507 edges: 22774
(250000, 20)


,mean,max
g_degree,50.700082,4.030000e+02
g_pagerank,0.000879,6.952264e-03
g_comm_size,675.207537,1.033000e+03
v_prior_tx,547.082996,6.416000e+03
v_secs_since_last,104448.540620,5.693681e+06
prior_fraud_card1,13.966568,3.140000e+02
prior_tx_card1,547.082996,6.416000e+03
prior_rate_card1,0.026833,1.000000e+00


In [ ]:
X_tr2 = pd.concat([X_tr, gf.iloc[:split]], axis=1)
X_te2 = pd.concat([X_te, gf.iloc[split:]], axis=1)

gbm2 = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                          subsample=0.8, colsample_bytree=0.6,
                          scale_pos_weight=spw, random_state=42, verbose=-1)
gbm2.fit(X_tr2, y_tr)
evaluate("LightGBM+Graph", gbm2.predict_proba(X_te2)[:, 1])
print(pd.DataFrame(results).T.round(4))

imp = pd.Series(gbm2.feature_importances_, X_tr2.columns).sort_values(ascending=False)
print("\nTop 10 features:\n", imp.head(10))
print("\nGraph features in top 30:", [c for c in imp.head(30).index if c in gf.columns])

joblib.dump(list(X_tr2.columns), BASE + "features_graph.pkl")
gf.to_parquet(BASE + "graph_features.parquet")

                Precision  Recall      F1  PR-AUC  ROC-AUC
LogReg             0.1243  0.7899  0.2148  0.4373   0.8567
RandomForest       0.2916  0.6311  0.3989  0.5178   0.8786
LightGBM           0.4301  0.6524  0.5185  0.6263   0.9235
LightGBM+Graph     0.4719  0.5936  0.5258  0.6012   0.9047

Top 10 features:
 TransactionAmt               686
prior_rate_card1             666
prior_rate_P_emaildomain     658
card1                        638
card2                        606
prior_rate_addr1             593
addr1                        586
prior_tx_addr1               515
prior_fraud_P_emaildomain    499
g_pagerank                   496
dtype: int32

Graph features in top 30: ['prior_rate_card1', 'prior_rate_P_emaildomain', 'prior_rate_addr1', 'prior_tx_addr1', 'prior_fraud_P_emaildomain', 'g_pagerank', 'prior_tx_comm_id', 'prior_fraud_comm_id', 'v_prior_tx', 'prior_rate_comm_id', 'prior_tx_P_emaildomain', 'v_secs_since_last', 'prior_fraud_addr1', 'g_degree', 'prior_fraud_card1', 'prior

In [ ]:
from sklearn.metrics import precision_recall_curve

def recall_at_precision(y, p, target):
    prec, rec, _ = precision_recall_curve(y, p)
    ok = rec[:-1][prec[:-1] >= target]
    return ok.max() if len(ok) else 0.0

p_base  = gbm.predict_proba(X_te)[:, 1]
p_graph = gbm2.predict_proba(X_te2)[:, 1]

print("Recall at fixed precision:")
for t in [0.5, 0.7, 0.8, 0.9]:
    print(f"  precision>={t}:  base={recall_at_precision(y_te,p_base,t):.3f}  "
          f"graph={recall_at_precision(y_te,p_graph,t):.3f}")

# Gain-based importance (more honest than split count)
gain = pd.Series(gbm2.booster_.feature_importance("gain"), X_tr2.columns)
gain = (gain / gain.sum()).sort_values(ascending=False)
print("\nTop 10 by gain:\n", gain.head(10).round(4))
print("\nShare of total gain from graph features:",
      round(gain[[c for c in gain.index if c in gf.columns]].sum(), 3))

joblib.dump(gbm2, BASE + "model_graph.pkl")
joblib.dump(gbm,  BASE + "model_base.pkl")

Recall at fixed precision:
  precision>=0.5:  base=0.611  graph=0.574
  precision>=0.7:  base=0.498  graph=0.477
  precision>=0.8:  base=0.445  graph=0.436
  precision>=0.9:  base=0.366  graph=0.364

Top 10 by gain:
 prior_rate_card1            0.2224
C14                         0.0451
D2                          0.0280
V294                        0.0273
TransactionAmt              0.0214
C4                          0.0197
prior_rate_P_emaildomain    0.0178
day                         0.0170
C13                         0.0151
C2                          0.0145
dtype: float64

Share of total gain from graph features: 0.402


['/content/drive/MyDrive/fraud detection project/model_base.pkl']

In [ ]:
import json
prec, rec, thr = precision_recall_curve(y_te, p_graph)
THRESH = float(thr[np.where(prec[:-1] >= 0.8)[0][0]])   # lowest threshold with precision >= 0.8
print("Alert threshold:", round(THRESH, 4))

json.dump({"features": list(X_tr2.columns), "threshold": THRESH},
          open(BASE + "config.json", "w"))
joblib.dump(gbm2, BASE + "model_graph.pkl")

rs = X_te2.copy()
rs["isFraud"] = y_te.values
rs["TransactionID"] = test["TransactionID"].values
rs.sample(1500, random_state=1).to_parquet(BASE + "replay_sample.parquet")

Alert threshold: 0.8162


DEPLOYMENT


In [ ]:
%%writefile /content/api.py
import json, sqlite3, time, joblib, pandas as pd
from typing import Dict, Optional
from fastapi import FastAPI
from pydantic import BaseModel

B = "/content/drive/MyDrive/fraud detection project/"
model = joblib.load(B + "model_graph.pkl")
cfg = json.load(open(B + "config.json"))
COLS, THR = cfg["features"], cfg["threshold"]

con = sqlite3.connect("/content/fraud.db", check_same_thread=False)
con.execute("""CREATE TABLE IF NOT EXISTS predictions
  (ts REAL, transaction_id INTEGER, probability REAL, decision TEXT, latency_ms REAL)""")
con.commit()

app = FastAPI(title="Fraud Detection API")

class Tx(BaseModel):
    transaction_id: int
    features: Dict[str, Optional[float]]

@app.get("/health")
def health():
    return {"status": "ok", "threshold": THR}

@app.post("/predict")
def predict(tx: Tx):
    t0 = time.time()
    row = pd.DataFrame([tx.features]).reindex(columns=COLS).astype(float)
    p = float(model.predict_proba(row)[:, 1][0])
    decision = "FRAUD_ALERT" if p >= THR else "OK"
    ms = (time.time() - t0) * 1000
    con.execute("INSERT INTO predictions VALUES (?,?,?,?,?)",
                (time.time(), tx.transaction_id, p, decision, ms))
    con.commit()
    return {"transaction_id": tx.transaction_id,
            "fraud_probability": round(p, 4), "decision": decision}

Writing /content/api.py


In [ ]:
!pip install -q fastapi uvicorn
import subprocess, sys, time, requests

log = open("/content/server.log", "w")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "api:app",
                           "--app-dir", "/content", "--port", "8000"],
                          stdout=log, stderr=log)
time.sleep(20)
print(requests.get("http://127.0.0.1:8000/health").json())

{'status': 'ok', 'threshold': 0.8162375992571764}


In [ ]:
import sqlite3, shutil
sample = pd.read_parquet(BASE + "replay_sample.parquet")
cfg = json.load(open(BASE + "config.json"))
recs = sample[cfg["features"]].head(300).to_dict("records")
ids = sample["TransactionID"].head(300).tolist()

for tid, r in zip(ids, recs):
    feats = {k: (None if pd.isna(v) else float(v)) for k, v in r.items()}
    requests.post("http://127.0.0.1:8000/predict",
                  json={"transaction_id": int(tid), "features": feats})

logdf = pd.read_sql("SELECT * FROM predictions", sqlite3.connect("/content/fraud.db"))
logdf = logdf.merge(sample[["TransactionID", "isFraud"]],
                    left_on="transaction_id", right_on="TransactionID")

print("Predictions logged:", len(logdf))
print("Latency ms  p50/p95:", logdf.latency_ms.quantile([.5, .95]).round(1).tolist())
print("Alerts:", (logdf.decision == "FRAUD_ALERT").sum(),
      "| actual fraud in replay:", logdf.isFraud.sum(),
      "| fraud caught:", ((logdf.decision == "FRAUD_ALERT") & (logdf.isFraud == 1)).sum())

shutil.copy("/content/fraud.db", BASE + "fraud.db")   # dashboard reads this

Predictions logged: 300
Latency ms  p50/p95: [45.3, 97.1]
Alerts: 7 | actual fraud in replay: 8 | fraud caught: 4


'/content/drive/MyDrive/fraud detection project/fraud.db'

In [ ]:
recs = sample[cfg["features"]].to_dict("records")
ids = sample["TransactionID"].tolist()
for tid, r in zip(ids, recs):
    feats = {k: (None if pd.isna(v) else float(v)) for k, v in r.items()}
    requests.post("http://127.0.0.1:8000/predict",
                  json={"transaction_id": int(tid), "features": feats})

logdf = pd.read_sql("SELECT * FROM predictions", sqlite3.connect("/content/fraud.db"))
logdf = logdf.drop_duplicates("transaction_id").merge(
    sample[["TransactionID", "isFraud"]], left_on="transaction_id", right_on="TransactionID")
a = logdf.decision == "FRAUD_ALERT"
print("alerts:", a.sum(), "| caught:", (a & (logdf.isFraud == 1)).sum(),
      "| total fraud:", logdf.isFraud.sum())
shutil.copy("/content/fraud.db", BASE + "fraud.db")

alerts: 30 | caught: 24 | total fraud: 53


'/content/drive/MyDrive/fraud detection project/fraud.db'

In [ ]:
import shutil
shutil.copy("/content/api.py", BASE + "api.py")
pd.DataFrame(results).T.round(4).to_csv(BASE + "results_table.csv")